# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.co

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'external company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'LinkedIn page', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook page',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [10]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 5 relevant links


{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook page',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [11]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links


{'links': [{'type': 'home page', 'url': 'https://huggingface.co/'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co'},
  {'type': 'status page', 'url': 'https://status.huggingface.co/'},
  {'type': 'GitHub', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Discord', 'url': 'https://huggingface.co/join/discord'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [12]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [13]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
NEW
GGML and llama.cpp join Hugging Face 🔥
Try HuggingChat Omni – Chat with AI 💬
Get started with Inference in seconds 🚀
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
Qwen/Qwen3.5-397B-A17B
Updated
1 day ago
•
105k
•
778
zai-org/GLM-5
Updated
8 days ago
•
174k
•
1.39k
MiniMaxAI/MiniMax-M2.5
Updated
5 days ago
•
123k
•
809
nvidia/personaplex-7b-v1
Updated
5 days ago
•
510k
•
2.09k
Nanbeige/Nanbeige4.1-3B
Updated
2 days ago
•
104k
•
628
Browse 2M+ models
Spaces
Running
Reachy
309
Reachy Phone Home
📱
309
Phone focus companion for Reachy Mini
Running
Featured
4.77k
Wan2.2 Animate
👁


In [14]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [15]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [16]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nNEW\nGGML and llama.cpp join Hugging Face 🔥\nTry HuggingChat Omni – Chat with AI 💬\nGet started with Inference in seconds 🚀\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nQwen/Qwen3.5-397B-A17B\nUpdated\n1 day ago\n•\n105k\n•\n779\nzai-org/GLM-5\nUpdated\n8 days ago\n•\n174k\n•\n1.39k\nMiniMaxAI/MiniMax-M2.5\nUpdated\n5 days ago\n•\n123k\n•\n809\nnvidia/personaplex-7b-v1\nUpdated\n5 days ago\n•\n510k\n•\n2.09k\nNanbeige/Nanbeige4.1-3B\

In [17]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [18]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 7 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is the premier AI community and collaboration platform dedicated to building the future of machine learning (ML). It serves as a central hub where the global ML community comes together to create, explore, and share hundreds of thousands of models, datasets, and AI-powered applications across multiple modalities including text, image, video, audio, and even 3D.

The platform empowers machine learning engineers, scientists, and enthusiasts to learn, collaborate, and showcase their work within an ethical and open AI environment. Hugging Face is known for its vast ecosystem—hosting over 2 million models, 500,000 datasets, and more than 1 million AI applications.

---

## Platform Features

- **Models**: Access and contribute to a library of more than 2 million ML models, including trending and cutting-edge architectures.
- **Datasets**: Browse an extensive collection of over 500,000 datasets updated frequently, spanning various research areas.
- **Spaces**: Discover and create interactive AI applications running on the platform.
- **Community & Collaboration**: Join an active, fast-growing global AI community dedicated to open source and ethical AI development.
- **Open Source Stack**: Build and deploy using Hugging Face’s open-source tools and libraries.
- **Multi-Modality Support**: Work fluidly across text, vision, audio, and 3D domains.

---

## Enterprise Solutions

Hugging Face offers tailored solutions for teams and enterprises to scale AI safely and efficiently:

- **Team Plans**: Starting at $20/user/month, ideal for collaboration and streamlined project workflows.
- **Enterprise Plans**: Custom contracts with advanced features such as enterprise-grade security, single sign-on (SSO), private dataset viewers, additional storage, granular access control, audit logs, and priority support.
- **Compute and Scalability**: Access advanced compute options, including enhanced ZeroGPU quota for increased performance and scalability.
- **Security & Compliance**: Organization-wide security policies, resource group management, centralized token control, and billing oversight.
- **Analytics & Billing**: Unified dashboards for usage analytics and control over spending limits with managed billing options.

---

## Company Culture & Community

- **Open and Ethical AI**: Hugging Face is deeply committed to building an open AI ecosystem that values transparency, collaboration, and ethical considerations.
- **Inclusivity and Growth**: The platform supports novices and experts alike, fostering learning and career growth opportunities for ML engineers and researchers.
- **Active Contributor Network**: Developers worldwide are encouraged to contribute models, datasets, and applications, helping the community innovate rapidly.
- **Portfolio Building**: Users can share their work publicly, build their ML profile, and gain industry recognition.

---

## Careers & Opportunities

Hugging Face is continuously growing and seeking passionate talent to join its mission of democratizing AI. They offer roles in engineering, research, product development, and community engagement. Employees thrive in a collaborative environment that values innovation, inclusivity, and impact.

---

## Why Choose Hugging Face?

- **World-Leading ML Hub**: The largest open platform for machine learning innovation.
- **Collaborative Ecosystem**: Connects community members and organizations to accelerate AI development.
- **Comprehensive Tools**: From individual experimentation to enterprise-scale deployments.
- **Commitment to Ethical AI**: Fostering responsible AI technologies and frameworks.
- **Supportive Enterprise Environment**: Advanced security, compliance, and premium support for organizations.

---

## Get Started

- Explore the unlimited open-source models, datasets, and AI apps available on Hugging Face.
- Join the vibrant AI community to collaborate and grow.
- Upgrade to Team or Enterprise plans for enhanced features and support.
- Visit [huggingface.co](https://huggingface.co) and sign up today.

---

**Hugging Face – The AI community building the future.**

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [19]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [20]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is the vibrant AI community and platform dedicated to building the future of machine learning. It is a collaborative space where developers, researchers, and organizations come together to create, share, and improve machine learning models, datasets, and AI applications. With over 2 million models and 500,000+ datasets available, Hugging Face is the go-to hub for innovation across various AI modalities including text, images, video, audio, and even 3D.

---

## What We Offer

### Models & Datasets
- Access and collaborate on a vast library of **2M+ models** and **500k+ datasets**.
- Work on trending models like Qwen3.5, GLM-5, MiniMaxAI, NVIDIA’s personaplex, and more.
- Discover datasets from multiple domains updated regularly by a passionate community.

### Spaces
- Host, share and run machine learning applications easily.
- Choose from thousands of AI-powered apps like video generation, image creation from text prompts, and demo playgrounds.
- Benefit from free platform access to multiple AI models with seamless scalability.

### Collaboration & Community
- Build, discover, and collaborate on machine learning projects effortlessly.
- Share your work publicly to build your ML portfolio and reputation.
- Engage with a global community committed to open source advancement in AI.

---

## Enterprise Solutions

Hugging Face also offers tailored **Enterprise and Team plans** designed to accelerate organizational AI adoption with robust security and performance features:

- **Team Plan ($20/user/month):** Quick setup for small to medium teams with collaborative tools, enhanced compute options, and priority support.
- **Enterprise Plan:** Flexible contracts with advanced security including Single Sign-On, audit logs, resource groups, token management, and more.
- Additional benefits include:
  - Private storage (1 TB per member + scalable)
  - Custom access controls and repository governance
  - Advanced analytics dashboard for usage tracking
  - ZeroGPU quota boosts for high-demand workloads
  - Managed billing and spending limits

Organizations can securely scale AI development with enterprise-grade tools and dedicated support from Hugging Face’s expert team.

---

## Pricing Overview

| Plan          | Key Features                                      | Cost                   |
| ------------- | ------------------------------------------------ | ---------------------- |
| **PRO**       | Increased storage, inference credits, ZeroGPU quota, Spaces hosting, PRO badge | $9 per month           |
| **Team**      | Collaboration tools, private storage, priority support | $20 per user per month  |
| **Enterprise**| Fully customizable with enterprise security, billing, analytics, and support | Contact sales          |

---

## Company Culture & Careers

Hugging Face fosters a culture of **openness, collaboration, and innovation**. It is a thriving community-driven company that believes in empowering people to build and deploy machine learning models with ease. The platform supports contributors from around the world providing a transparent and inclusive environment where ideas can grow and impact the future of AI.

**Join Hugging Face** to work with cutting-edge technologies in NLP, computer vision, and multimodal AI. They offer career opportunities for engineers, researchers, community managers, and enterprise solutions specialists passionate about shaping the next wave of AI technology.

---

## Why Choose Hugging Face?

- **Largest collaborative AI hub:** Host and explore millions of models and datasets with a global community.
- **Open source & innovation:** Leverage the HF open-source stack to move faster and explore new AI modalities.
- **Enterprise-ready:** Secure, scalable, and supported platform tailored for team and organizational needs.
- **Build your profile:** Showcase your AI projects, contribute to the ecosystem, and be part of a groundbreaking AI future.

---

## Get Started

Whether you're an individual researcher, developer, or large organization, Hugging Face offers the tools, community, and infrastructure to accelerate your AI projects. 

**Sign up today** and join the AI community building the future.

[Explore Hugging Face](https://huggingface.co)

---

### Contact

- **Website:** https://huggingface.co  
- **Enterprise Sales:** Reach out via the website for custom enterprise plans  
- **Community & Support:** Active forums and documentation available online  

---

*Hugging Face – The AI community building the future.*

In [21]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

# stream_brochure("HuggingFace", "https://huggingface.co")

## Using the Playwright Implementation

In [38]:
from scraper import playwright_fetch_website_contents, playwright_fetch_website_links
from pydantic import BaseModel

import nest_asyncio
nest_asyncio.apply() 

In [23]:
url = "https://huggingface.co"

In [ ]:
hf_links = playwright_fetch_website_links(url)
len(hf_links)

31

In [27]:
hf_links[:5]

['https://huggingface.co/',
 'https://huggingface.co/models',
 'https://huggingface.co/datasets',
 'https://huggingface.co/spaces',
 'https://huggingface.co/docs']

In [28]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [29]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = playwright_fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [30]:
print(get_links_user_prompt(url))


Here is the list of links on the website https://huggingface.co -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://huggingface.co/
https://huggingface.co/models
https://huggingface.co/datasets
https://huggingface.co/spaces
https://huggingface.co/docs
https://huggingface.co/enterprise
https://huggingface.co/pricing
https://huggingface.co/login
https://huggingface.co/join
https://huggingface.co/blog/ggml-joins-hf
https://huggingface.co/Qwen/Qwen3.5-397B-A17B
https://huggingface.co/zai-org/GLM-5
https://huggingface.co/MiniMaxAI/MiniMax-M2.5
https://huggingface.co/nvidia/personaplex-7b-v1
https://huggingface.co/Nanbeige/Nanbeige4.1-3B
https://huggingface.co/spaces/itsMarco-G/reachy_phone_home
https://huggingface.co/spaces/Wan-AI/Wan2.2-Animate
https://huggingface.co/spaces/r3gm/wan2-2-fp8da-aoti-preview

In [54]:
MODEL = "gpt-5-nano"

class Link(BaseModel):
    type: str
    url: str

class PageLinks(BaseModel):
    links: list[Link]


def select_relevant_links(url):
    response = openai.responses.parse(
        model=MODEL,
        input=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        # text_format={"type": "json_object"}
        text_format=PageLinks
    )
    result = response.output_parsed
    # links = json.loads(result)
    links = result.model_dump()
    print(f"Found {len(links['links'])} relevant links")
    return links

In [55]:
result = select_relevant_links(url)

Found 13 relevant links


In [56]:
result

{'links': [{'type': 'homepage', 'url': 'https://huggingface.co/'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'partner page', 'url': 'https://huggingface.co/allenai'},
  {'type': 'partner page', 'url': 'https://huggingface.co/facebook'},
  {'type': 'partner page', 'url': 'https://huggingface.co/amazon'},
  {'type': 'partner page', 'url': 'https://huggingface.co/google'},
  {'type': 'partner page', 'url': 'https://huggingface.co/Intel'},
  {'type': 'partner page', 'url': 'https://huggingface.co/microsoft'},
  {'type': 'partner page', 'url': 'https://huggingface.co/grammarly'},
  {'type': 'partner page', 'url': 'https://huggingface.co/Writer'}]}

In [60]:
def fetch_page_and_all_relevant_links(url):
    contents = playwright_fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += playwright_fetch_website_contents(link["url"])
    return result

In [61]:
page_relevant_links = fetch_page_and_all_relevant_links(url)

Found 16 relevant links


In [62]:
print(page_relevant_links)

## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Pricing
Log In
Sign Up
NEW
GGML and llama.cpp join Hugging Face 🔥
Try HuggingChat Omni – Chat with AI 💬
Get started with Inference in seconds 🚀
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending onthis week
Models
Qwen/Qwen3.5-397B-A17B
Updated 1 day ago
•
133k
•
781
zai-org/GLM-5
Updated 8 days ago
•
177k
•
1.39k
MiniMaxAI/MiniMax-M2.5
Updated 5 days ago
•
173k
•
811
nvidia/personaplex-7b-v1
Updated 5 days ago
•
539k
•
2.09k
Nanbeige/Nanbeige4.1-3B
Updated 2 days ago
•
130k
•
630
Browse 2M+ models
Spaces
Reachy Phone Home
📱
310
Phone focus companion for Reachy Mini
Wan2.2 Animate
👁
4.77k
Wan2.2 Animate
Wan2.2 14B Preview
🐌
829
generate a video from an image with a text prompt
Demo Playground
⚡
372
Free platform to access multiple A

In [63]:
len(page_relevant_links)

65510

In [65]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [64]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt
    return user_prompt

In [66]:
print(get_brochure_user_prompt("HuggingFace", "https://huggingface.co"))

Found 9 relevant links

You are looking at a company called: HuggingFace
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.


## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Pricing
Log In
Sign Up
NEW
GGML and llama.cpp join Hugging Face 🔥
Try HuggingChat Omni – Chat with AI 💬
Get started with Inference in seconds 🚀
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending onthis week
Models
Qwen/Qwen3.5-397B-A17B
Updated 1 day ago
•
133k
•
781
zai-org/GLM-5
Updated 8 days ago
•
177k
•
1.39k
MiniMaxAI/MiniMax-M2.5
Updated 5 days ago
•
173k
•
811
nvidia/personaplex-7b-v1
Updated 5 days ago
•
539k
•
2.09k
Nanbeige/Nanbeige4.1-3B
Updated 2 days ago
•
130k
•
630
Browse 2M+ models


In [69]:
def create_brochure(company_name, url):
    brochure_user_prompt = get_brochure_user_prompt(company_name, url)
    response = openai.responses.create(
        model="gpt-4.1-mini",
        input=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": brochure_user_prompt}
        ],
    )
    result = response.output_text
    display(Markdown(result))

In [70]:
create_brochure("HuggingFace", "https://huggingface.co")

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/Users/patrickwalukagga/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/asyncio/events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x10ab87140> is already entered


Found 6 relevant links


Task was destroyed but it is pending!
task: <Task pending name='Task-847' coro=<_async_in_context.<locals>.run_in_context() done, defined at /Users/patrickwalukagga/Projects/personal/llm_engineering/.venv/lib/python3.12/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-849' coro=<Kernel.shell_main() running at /Users/patrickwalukagga/Projects/personal/llm_engineering/.venv/lib/python3.12/site-packages/ipykernel/kernelbase.py:590> cb=[Task.__wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /Users/patrickwalukagga/Projects/personal/llm_engineering/.venv/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py:563]>
/Users/patrickwalukagga/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/json/decoder.py:354: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  obj, end = self.scan_once(s, idx)
Task was destroyed but it is pending!
task: <Task pending name='Task-849' coro=<Kernel.shell_main() running at /Users/patrickw

# Hugging Face - The AI Community Building the Future

---

## About Hugging Face

Hugging Face is a pioneering platform and community at the forefront of the AI revolution, enabling the machine learning community to collaborate on models, datasets, and applications. As the home of open-source machine learning (ML), Hugging Face empowers ML engineers, scientists, and end-users to create, share, and discover transformative AI technology.

The platform hosts over 2 million models and 500k datasets across modalities including text, images, video, audio, and 3D. Hugging Face offers a range of open-source libraries and tools such as Transformers, Diffusers, Tokenizers, and Accelerate — foundational building blocks that enable fast, efficient development and deployment of AI applications.

---

## Company Culture

Hugging Face thrives on openness, collaboration, and community. It centers efforts on empowering the next generation of AI builders through education, open-source projects, and a vibrant global community. The company champions ethical AI development, encouraging sharing, learning, and transparent innovation.

Key cultural pillars include:

- **Community Collaboration:** A hub for ML practitioners worldwide to share and improve upon AI models and datasets.
- **Open-Source Leadership:** Maintaining popular libraries with extensive contributions from talented scientists and engineers.
- **Innovation Education:** Offering comprehensive courses on topics such as large language models, robotics, audio, computer vision, and reinforcement learning to accelerate learning.
- **Ethical & Responsible AI:** Commitment to building an open and ethical AI future together.

---

## What Hugging Face Offers

### Platform Features

- **Hub:** Centralized for exploring, experimenting, and collaboration on open-source machine learning projects.
- **Models & Datasets:** Access to over 2 million models and 500k+ datasets, continuously updated and contributed by a global community.
- **Spaces:** Share AI applications and demos with flexible, on-demand compute including GPUs for rapid prototyping and hosting.
- **Inference API:** Unified access to 45,000+ models from major AI providers without service fees.
- **Compute Solutions:** Deploy optimized endpoints and scale AI workloads with enterprise-grade security and advanced access controls.

---

## Customers & Users

Over 50,000 organizations utilize Hugging Face’s platform, ranging from non-profits to leading tech giants like:

- **Google, Microsoft, Amazon, Meta, Intel**
- **NVIDIA, Salesforce, Apple, IBM**
- **Grammarly, Roblox, Airbnb, Toyota Research Institute**

These organizations rely on Hugging Face’s tools for research, production AI deployment, innovation, and collaboration.

---

## Enterprise & Teams

Hugging Face offers tailored enterprise-grade solutions designed to scale AI initiatives at organizations:

- **Security:** Single Sign-On (SSO), audit logs, advanced access and token management.
- **Collaboration:** Resource groups for granular permissions, private dataset viewers, and repository usage analytics.
- **Compute:** Advanced compute offerings, ZeroGPU quota boosts, dedicated inference endpoints.
- **Support:** Priority support and managed billing options.

Pricing starts at $20/user/month for teams, with custom plan options for enterprises to meet unique requirements.

---

## Pricing Highlights

- **Free Tier:** Access to the Hugging Face Hub with limitless exploration and community engagement.
- **PRO Account:** $9/month unlocking enhanced storage, inference credits, priority usage, blog publishing, and early access features.
- **Team Plan:** Starting at $20/user/month with SSO, audit logs, and advanced security controls.
- **Enterprise Plans:** From $50/user/month including managed billing, custom SLAs, and enhanced compliance.

Additional cost-effective storage and premium GPU-powered Spaces hosting options are available.

---

## Careers

Join Hugging Face to be part of the AI community transforming technology. The company actively recruits talents passionate about machine learning, open source, and ethical AI innovation. 

Careers focus on engineering, research science, product development, and community engagement roles, placing individuals at the cutting edge of AI technology.

---

## Learn With Hugging Face

Hugging Face offers extensive learning resources and courses covering:

- Large Language Models (LLMs)
- Robotics & AI Agents
- Deep Reinforcement Learning
- Computer Vision & 3D Machine Learning
- Audio Processing with Transformers
- Diffusion Models
- Open-source AI cookbook and much more

These courses leverage Hugging Face’s robust ecosystem and community knowledge, ideal for newcomers and experts looking to deepen their skills.

---

## Social & Community

Stay connected and participate in conversations via:

- GitHub
- Twitter
- LinkedIn
- Discord forum

The regularly updated blog features research breakthroughs, community stories, tutorials, and case studies.

---

## Brand and Identity

- Colors: Bright yellow (#FFD21E), orange (#FF9D00), and gray (#6B7280).
- Logo assets available in vector and raster formats show Hugging Face’s friendly and forward-thinking personality.

---

## Summary

Hugging Face is the go-to platform and community for anyone passionate about AI and machine learning. With leading open-source tools, a global network of collaborators, and flexible enterprise solutions, it’s reshaping how AI models and datasets are built, shared, and deployed worldwide.

Start building and collaborating with Hugging Face to join a movement that is defining the future of AI.

---

**Explore Hugging Face today!**

Website: https://huggingface.co  
Twitter: @huggingface  
GitHub: https://github.com/huggingface  
Join the community and build the future of AI.

In [71]:
def stream_brochure(company_name, url):
    brochure_user_prompt = get_brochure_user_prompt(company_name, url)
    stream = openai.responses.create(
        model="gpt-4.1-mini",
        input=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": brochure_user_prompt}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for event in stream:
        if event.type == "response.output_text.delta":
            response += event.delta
            update_display(Markdown(response), display_id=display_handle.display_id)

In [72]:
company_name = "HuggingFace"
url = "https://huggingface.co"
stream_brochure(company_name, url)

Found 10 relevant links


# Hugging Face Brochure  
*The AI community building the future*

---

## About Hugging Face  
Hugging Face is the premier **collaboration platform for the machine learning community**, empowering engineers, scientists, and AI enthusiasts worldwide to learn, innovate, and share open-source machine learning tools. At the core of the AI revolution, Hugging Face fosters **an open and ethical AI future** through its vibrant community and extensive ecosystem of models, datasets, and applications.

---

## What We Offer   

### The Hugging Face Hub  
- Centralized platform hosting **2 million+ machine learning models** and **500,000+ datasets** across **text, image, video, audio, and 3D modalities**  
- Support for multiple ML libraries such as Transformers (156K+ models), Diffusers (32K+ models), Tokenizers, PEFT, and more  
- Collaboration tools for sharing and version controlling models, datasets, and applications (Spaces)  
  
### Spaces  
- AI App Directory featuring apps for image generation, video synthesis, speech synthesis, code generation, medical imaging, and more  
- Easy deployment with customizable compute resources including GPUs for efficient model hosting and inference  
- Community showcases popular projects like text-to-video creators, voice cloning apps, and interactive AI demos  

### Enterprise Solutions  
- Scalable, enterprise-grade platform with **security, access control, and dedicated support**  
- Features such as **Single Sign-On, Audit Logs, Resource Groups, Private Dataset Viewer**, and analytics  
- Flexible pricing starting at $20/user/month with managed billing and legal compliance (GDPR, SOC 2 Type 2)  
- Trusted by **50,000+ organizations** including Google, Microsoft, NVIDIA, Meta, Amazon, IBM, Salesforce  

### Compute & Inference  
- Optimized inference endpoints with multi-cloud support (AWS, GCP, Azure)  
- GPU/TPU infrastructure with flexible pricing (from $0.60/hour for GPU instances)  
- Unified API access to 45,000+ models through top inference providers with no service fees  

---

## Our Customers  
Hugging Face powers AI innovation for a wide range of organizations:  

- **Tech Giants:** Google (1,080+ models), Microsoft, Meta AI (2,300+ models), NVIDIA, Amazon  
- **Enterprises:** Salesforce, IBM Research, Intel, Apple, Toyota Research, Palo Alto Networks  
- **Teams & Startups:** Grammarly, JetBrains, Writer, Red Hat AI, FuriosaAI, TNG Technology Consulting  
- **Research & Non-profits:** Allen Institute for AI (AI2), UC Berkeley, AI at Meta, IBM Granite  

---

## Company Culture  
- **Open & Ethical AI:** Committed to building transparent, accessible AI technologies powered by the global ML community  
- **Collaboration & Sharing:** Encouraging community contributions, peer reviews, and shared learning through forums, blogs, and Discord  
- **Innovation Driven:** With a talented science team at the frontier of AI research and active engagement in open source projects  
- **Support & Learning:** Extensive educational resources, including courses on large language models, robotics, diffusion models, computer vision, and reinforcement learning  
- **Diversity & Inclusion:** Welcoming contributors worldwide, fostering growth for the next generation of ML engineers and researchers  

---

## Careers at Hugging Face  
Join a leading AI company shaping the future of machine learning!  

- Roles in AI research, machine learning engineering, software development, community management, and enterprise support  
- Work alongside AI experts building tools and libraries used globally  
- Opportunities to contribute to open-source projects that impact millions  
- Collaborative, flexible work environment valuing innovation and continuous learning  

Explore current openings and apply at: [huggingface.co/careers](https://huggingface.co/careers)  

---

## Why Choose Hugging Face?  
- **Largest Open-Source AI Community:** A vast array of models, datasets, and tools at your fingertips  
- **Unmatched Flexibility:** Support for all major ML frameworks and modalities with enterprise-grade security  
- **Proven at Scale:** Trusted by innovators from startups to tech giants worldwide  
- **Accelerate AI Development:** From free plan access to paid compute, scale seamlessly  
- **Build & Showcase Your Portfolio:** Share your work, gain recognition, and collaborate with AI experts  

---

## Get Started  
- Explore models, datasets, and AI applications at [huggingface.co](https://huggingface.co)  
- Sign up and create your free profile to start building and sharing ML projects  
- Upgrade with PRO accounts or enterprise plans to unlock advanced features and compute  
- Join the community on Twitter, Discord, LinkedIn, and GitHub  

---

**Hugging Face – Building the future of AI, together.**

In [73]:
company_name = "OpenAI"
url = "https://openai.com"
stream_brochure(company_name, url)

Found 14 relevant links


# OpenAI Brochure

---

## About OpenAI

OpenAI is a leading AI research and deployment company founded in 2015, dedicated to ensuring that artificial general intelligence (AGI) benefits all of humanity. AGI refers to highly autonomous AI systems that outperform humans at most economically valuable work. OpenAI’s vision is to build safe and beneficial AGI or to empower others who achieve this outcome.

OpenAI consists of the nonprofit OpenAI Foundation and the for-profit OpenAI Group, operating as a public benefit corporation. The Foundation governs the Group, advancing the mission through a combined impact.

---

## Mission & Charter

- **Humanity First:** Developing AI to elevate people and society.
- **Broadly Distributed Benefits:** Ensuring AGI is used widely and fairly.
- **Long-term Safety:** Committed to research that ensures AI safety and adopting safety measures across the AI community.
- **Technical Leadership:** Leading in AI advancement while considering societal impacts.
- **Cooperative Orientation:** Collaborating globally with research and policy institutions.
- **Fiduciary Duty:** Prioritizing humanity’s best interests to minimize conflicts.

The Charter guides OpenAI in ethically and responsibly developing powerful AI technologies.

---

## Products & Solutions

### AI Technologies
- **GPT-5 and GPT-5.2:** Advanced language models with multimodal capabilities (text, image, audio, vision) designed for professional, reliable output.
- **Codex:** AI coding assistants that write, debug, and optimize software.
- **Sora:** A natively multimodal AI model providing photorealistic and accurate outputs in images and video, with capabilities for audio and speech.
- **OpenAI o Series:** Advanced reasoning AI models specialized in STEM problem-solving using chain-of-thought methodologies.

### Business Solutions
- **ChatGPT for Business & Enterprise:** AI-powered tools for productivity, secure with enterprise-grade compliance including GDPR, CCPA, HIPAA, SOC 2 Type 2.
- **API Platform:** Fastest, most powerful API access to frontier AI models enabling custom AI products, natural language processing, content generation, customer support automation, data analysis, and coding acceleration.
- Integrations with common business tools like Google Drive, SharePoint, GitHub, and Dropbox.

### Industry Applications
- Financial services, healthcare, life sciences, retail, education, and more.
- AI agents that perform autonomous tasks and research.
- Safe deployment with no customer data used for training (upon request).

---

## Research and Safety

OpenAI pioneers research to safely approach AGI, focusing on:
- Alignment techniques, learning from human feedback.
- Testing and improving safeguards with internal evaluations and external experts.
- Addressing risks related to bias, privacy, misinformation, and misuse.
- Developing standards and collaborating with the global AI community on safety practices.
- Sharing public goods, while balancing the need for security in AI research dissemination.

Significant research milestones include improving scientific research capabilities, optimizing performance on real-world tasks, and enhancing AI's ability to reason through complex problems.

---

## Customers & Community

OpenAI serves thousands of businesses and developers worldwide, powering apps from startups to enterprises such as Notion and other industry innovators. The company supports AI-driven product creation, business workflow enhancement, and advanced research acceleration for global impact.

OpenAI for Startups offers resources, community, and specialized content to empower ambitious builders and founders through every stage of development.

---

## Careers & Culture

### Culture and Values:
- **Humanity First:** Focus on social good and ethical AI.
- **Act with Humility:** Staying open to feedback and new ideas.
- **Feel the AGI:** Deep responsibility combined with creativity in building transformative technology.
- **Ship Joy:** Optimism in delivering products that improve lives.
- **Find a Way:** Empowering teams to innovate with agency.
- **Creativity over Control:** Favoring flexible, principled problem-solving.
- **Update Quickly:** Agile adaptation based on new information.
- **Intense Focus:** Resilience and clarity toward meaningful impact.

### Benefits & Development:
- Comprehensive health, dental, vision, mental health, fertility, and family planning coverage.
- Generous parental leave and flexible remote work policies.
- Company-sponsored retirement plans and global travel insurance.
- Daily meals, wellness coaching, paid time off, and learning stipends.
- Employee resource groups and regular team celebrations.

### Opportunities:
- OpenAI Residency program supports researchers and engineers new to AI.
- Internships, full-time roles, and early-career positions fostering diverse backgrounds.
- Active recruitment across research, engineering, policy, business, and safety.

---

## Branding and Partnership

OpenAI’s branding reflects a balance of technological precision and human warmth, embodied in their signature wordmark and the Blossom logo symbolizing the synergy of humanity and technology. Partners must adhere to strict guidelines ensuring consistent, respectful representation.

---

## Contact & Get Started

- **For Businesses:** Contact OpenAI’s sales team to explore enterprise AI solutions, integrations, and custom AI model development.
- **For Developers:** Access the API platform with simple pricing, documentation, and community support.
- **For Startups:** Engage with the OpenAI Startup program to leverage resources and community for AI innovation.

Explore products, research publications, safety reports, and latest news directly at OpenAI’s website.

---

OpenAI — Advancing the frontier of artificial intelligence to benefit all of humanity.

---

*For more information, visit: [openai.com](https://www.openai.com)*

In [74]:
company_name = "Sunbird AI"
url = "https://sunbird.ai"
stream_brochure(company_name, url)

Found 18 relevant links


# Sunbird AI Brochure

---

## About Sunbird AI

Sunbird AI is a pioneering artificial intelligence research organization dedicated to applying AI technologies to solve African problems. Based in Kampala, Uganda, Sunbird AI focuses on creating scalable and impactful AI solutions tailored to the unique challenges faced by African communities. Their work spans across language technology, environmental sensing, citizen engagement, electrification, and health, utilizing AI for social good and sustainable development.

---

## Vision & Mission

Sunbird AI aims to harness the power of computational intelligence and machine learning to benefit humanity, particularly in developing African contexts. Their mission emphasizes:

- Developing AI tools relevant to African languages and communities.
- Addressing social, environmental, and economic challenges through evidence-informed decisions.
- Advancing inclusive AI technologies that empower marginalized populations.
- Promoting home-grown innovations and capacity building in AI research.

---

## Flagship Project: Sunflower Multilingual Assistant

- **Multilingual AI assistant** supporting 31 Ugandan languages.
- Provides **accurate translations**, explanations, summaries, and conversational capabilities.
- Enables cross-lingual tasks, empowering non-English speakers with accessible AI-driven communication.
- Offers an **API** for integration into broader applications.
- Developed with a focus on **local cultural and societal relevance**.
- Freely available to users with an open approach to data and model access.

---

## Key Research Areas and Projects

### African Language Technology

- Development of the **Sunbird Translate system** for translating between English and multiple Ugandan languages such as Acholi, Ateso, Luganda, Lugbara, and Runyankole.
- Creation of the **SALT dataset**, a multi-way parallel corpus tailored to locally relevant topics like healthcare, agriculture, and society.
- Speech technology including **Text-to-Speech (TTS)** for Luganda based on crowdsourced voice data — the only TTS system available for any Ugandan language.
- Promoting citizen feedback systems that allow voice-based communication in multiple local languages, improving public service delivery and inclusive engagement.

### Environmental Sensing & Noise Pollution

- Collaborative work with Kampala and Entebbe authorities to monitor **noise pollution**, a serious but under-represented health hazard in urban Africa.
- Deployment of custom hardware and real-time noise data analysis to aid urban planning and health improvement efforts.
- Data and results openly available for research and public use.

### Green Electrification Planning

- Partnership with the German Agency for International Cooperation (GIZ) and Uganda’s Ministry of Energy on the **National Electrification Strategy**.
- AI-driven site identification tool for renewable energy projects, focusing on solar, hydro, biomass, and wind energy to increase electricity access.
- Use of satellite and remote sensing data to optimize electrification plans, forecast urban growth, and improve service delivery in rural areas.
  
### Citizen Feedback & Social Impact

- Deployment of AI technologies that collect and analyze citizen feedback via voice and messaging, making governance more responsive.
- Partnerships with Uganda’s Ministry of ICT, TRAC FM radio, and SEMA Uganda to enhance public service and policy through local language AI tools.
- Tools enable marginalized voices to be heard in public discourse and government programs like the Parish Development Model.

---

## Leadership & Team

Sunbird AI boasts a highly qualified and diverse team combining academic excellence, industry experience, and social impact dedication:

- **Engineer Bainomugisha** – Founding Director, Associate Professor at Makerere University, leader of AI and IoT social impact initiatives.
- **Ernest Mwebaze** – Executive Director, with substantive roles at Google AI Research and UN Pulse Lab; expert in AI for development.
- **John Quinn** – Director and Senior Software Engineer at Google Ghana, expert in satellite imagery and AI for community analysis.
- A dedicated team of software engineers, researchers, fellows, and operations staff with expertise in machine learning, data science, AI model deployment, development communications, and project management.
- The advisory board includes leaders in sustainable development and data ecosystems like **Davis Adieno**, Chair and Director at the Global Partnership for Sustainable Development Data.

---

## Company Culture

Sunbird AI fosters an environment of:

- **Collaboration across disciplines** combining AI research, social sciences, and community engagement.
- **Mission-driven innovation**, focused on technology that produces tangible social benefits.
- Commitment to **open science**, sharing data, code, and models freely to empower broader communities.
- Support for **early-career researchers and students** through its competitive Fellows Program, offering hands-on AI project experience aligned with societal impact.
- Emphasis on **local capacity building and inclusion**, supporting African-led AI research and innovation.

---

## Careers & Fellowship Program

Sunbird AI offers opportunities including:

- A competitive 6-month **Fellows Program** for researchers and PhD students to collaborate on projects that create social impact through AI.
- Fellowships can be full-time/part-time and are flexible with options for remote or on-site work.
- Fellows receive mentorship from experienced AI researchers and work on projects aligned with Sunbird AI’s mission.
- Applications for the fellowship require a project plan, motivation letter, and optional supplementary materials.
- Join a passionate team working to leverage AI to improve African societies.

---

## Who Sunbird AI Serves

- **Local government authorities** seeking AI solutions for public service delivery and urban planning.
- **Health organizations** improving maternal and sexual reproductive health through AI-powered data.
- **Energy planners** implementing national strategies for renewable energy and electrification.
- **Community radio stations** and citizen organizations to facilitate inclusive and multilingual feedback.
- **Developing country societies**, especially those with limited internet and literacy, empowered by voice and language technologies.

---

## Contact & Engagement

- **Office Address:** Plot 15, Naguru East Road, Kampala, Uganda
- **Mailing:** PO Box 11296 Kampala, Uganda
- **Email:** info@sunbird.ai
- Follow Sunbird AI on **Twitter**, **Youtube**, and **Medium**
- Engage with the team for advice or collaborations through bookings available on their community page.
- Papers, APIs, and models are openly accessible for developers and organizations interested in expanding African language AI technologies.

---

For more information or collaboration opportunities, visit [sunbird.ai](https://sunbird.ai) or contact info@sunbird.ai.

---

**Sunbird AI – Advancing Artificial Intelligence Research for African Solutions**

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>